# 01 · Análise exploratória

Este notebook organiza a EDA em cinco etapas:

1. entendimento e qualidade dos dados, com publicação do dicionário;
2. análise univariada das variáveis quantitativas, qualitativas e temporal;
3. relações entre as features, excluindo `Load_Type`, definido como target;
4. diagnósticos de assimetria, valores extremos e padrões temporais;
5. segmentação exploratória em três regimes e detecção não supervisionada de anomalias.

## Como as técnicas respondem ao problema

A pergunta central é se as medições de energia apresentam estrutura compatível com três regimes operacionais, sem usar os rótulos de Load_Type para ensinar essa estrutura aos modelos. Cada técnica possui um papel específico:

| Técnica | Para que serve neste notebook | Decisão metodológica |
|---|---|---|
| Estatísticas descritivas e dicionário | Verificar escala, domínio, qualidade e significado das variáveis | Aplicadas antes de qualquer transformação |
| Histogramas e Freedman–Diaconis | Examinar forma, concentração, caudas e multimodalidade | Aplicados às variáveis contínuas |
| Boxplot e IQR | Sinalizar valores extremos de maneira robusta | Diagnóstico; não remove registros automaticamente |
| Pearson e Spearman | Comparar associações lineares e monotônicas | Usados em conjunto para evitar leitura exclusivamente linear |
| Perfis dia–hora | Identificar ciclos temporais que podem explicar regimes | Mantêm interpretação operacional do tempo |
| PCA | Resumir variação global em poucos eixos lineares | Projeção e interpretação; não define clusters |
| t-SNE | Visualizar vizinhanças locais e possíveis separações não lineares | Apenas visualização; não seleciona K |
| K-Means | Produzir uma segmentação simples e interpretável em três grupos | Baseline principal com K fixado em 3 |
| Cotovelo e silhouette | Verificar compactação e separação para diferentes valores de K | Diagnósticos; não substituem a hipótese do problema |
| GMM | Representar três grupos com sobreposição e pertencimento probabilístico | Contraponto ao K-Means com três componentes |
| AIC e BIC | Medir ajuste versus complexidade no GMM | Diagnósticos de subestrutura, não escolha dos regimes |
| Isolation Forest | Sinalizar observações globalmente incomuns | Complementa os três regimes; não cria nova classe |
| Métodos adiados | Registrar alternativas, limitações e condições de uso futuro | Não executados nesta EDA |

Essa separação de papéis evita antecipar avaliação supervisionada e impede que uma projeção visual ou uma única métrica seja tratada como confirmação dos regimes.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import OneHotEncoder, StandardScaler

COLORBLIND_PALETTE = sns.color_palette("colorblind")
PRIMARY_COLOR = COLORBLIND_PALETTE[0]
ACCENT_COLOR = COLORBLIND_PALETTE[1]
ANOMALY_COLOR = COLORBLIND_PALETTE[3]
sns.set_theme(
    context="notebook",
    style="whitegrid",
    palette=COLORBLIND_PALETTE,
    rc={
        "axes.titleweight": "bold",
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "grid.alpha": 0.25,
        "legend.frameon": True,
        "figure.dpi": 110,
    },
)
pd.set_option("display.max_columns", None)

IMAGES_DIR = Path("../images")
TABLES_DIR = Path("../tables")
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

def save_figure(figure: plt.Figure, filename: str) -> None:
    output_path = IMAGES_DIR / filename
    figure.savefig(output_path, dpi=180, bbox_inches="tight", facecolor="white")
    print(f"Imagem salva em: {output_path.resolve()}")

## 1. Entendimento e qualidade dos dados

**Para que serve:** estabelecer se a base pode sustentar as análises seguintes sem erros de tipo, domínio ou completude.

O primeiro bloco verifica o contrato básico da base: dimensões, tipos, duplicidades, valores ausentes e domínio observado. O dicionário é construído a partir do CSV efetivamente carregado e publicado em artifacts/data/processed, evitando divergência entre documentação e dados. Esta etapa não procura regimes; ela garante que padrões posteriores não sejam artefatos de leitura ou qualidade.

In [ ]:
DATA_PATH = Path("../data/raw/Steel_industry_data.csv")
TARGET_COLUMN = "Load_Type"
DATE_COLUMN = "date"

df = pd.read_csv(DATA_PATH)
df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], dayfirst=True, errors="raise")
EXPECTED_N_GROUPS = len(df[TARGET_COLUMN].unique())
assert EXPECTED_N_GROUPS == 3, "O problema atual pressupõe exatamente três regimes."
print(f"Hipótese estrutural do problema: {EXPECTED_N_GROUPS} regimes.")
df.head()

### 1.1 Enquadramento do problema e uso restrito do target

A cardinalidade de Load_Type define a hipótese estrutural de três regimes. Somente len(df.Load_Type.unique()) é utilizado para fixar esse número; os valores das classes não entram nas features, no treinamento, nas cores dos clusters nem nas métricas de avaliação.

A frequência univariada de Load_Type permanece na EDA apenas para documentar o conjunto. Relações feature–target, comparação cluster–classe, ARI, acurácia, matriz de confusão e modelos de classificação ficam deliberadamente reservados para etapas posteriores.

In [ ]:
dictionary_metadata = [
    {"coluna": "date", "papel": "identificador temporal", "tipo_uci": "Date", "unidade": "-", "descricao": "Data e hora da medição, em intervalos de 15 minutos."},
    {"coluna": "Usage_kWh", "papel": "feature", "tipo_uci": "Continuous", "unidade": "kWh", "descricao": "Consumo de energia elétrica da indústria no intervalo."},
    {"coluna": "Lagging_Current_Reactive.Power_kVarh", "papel": "feature", "tipo_uci": "Continuous", "unidade": "kVarh", "descricao": "Energia reativa associada à corrente atrasada."},
    {"coluna": "Leading_Current_Reactive_Power_kVarh", "papel": "feature", "tipo_uci": "Continuous", "unidade": "kVarh", "descricao": "Energia reativa associada à corrente adiantada."},
    {"coluna": "CO2(tCO2)", "papel": "feature", "tipo_uci": "Continuous", "unidade": "ppm (conforme UCI)", "descricao": "Emissão de dióxido de carbono associada ao consumo de energia."},
    {"coluna": "Lagging_Current_Power_Factor", "papel": "feature", "tipo_uci": "Continuous", "unidade": "%", "descricao": "Fator de potência da corrente atrasada."},
    {"coluna": "Leading_Current_Power_Factor", "papel": "feature", "tipo_uci": "Continuous", "unidade": "%", "descricao": "Fator de potência da corrente adiantada."},
    {"coluna": "NSM", "papel": "feature", "tipo_uci": "Integer", "unidade": "s", "descricao": "Número de segundos transcorridos desde a meia-noite."},
    {"coluna": "WeekStatus", "papel": "feature", "tipo_uci": "Categorical", "unidade": "-", "descricao": "Indica dia útil ou fim de semana."},
    {"coluna": "Day_of_week", "papel": "feature", "tipo_uci": "Categorical", "unidade": "-", "descricao": "Dia da semana da medição."},
    {"coluna": "Load_Type", "papel": "target", "tipo_uci": "Categorical", "unidade": "-", "descricao": "Categoria do nível de carga elétrica."},
]

def observed_domain(series: pd.Series) -> str:
    if pd.api.types.is_datetime64_any_dtype(series):
        return f"{series.min():%d/%m/%Y %H:%M} a {series.max():%d/%m/%Y %H:%M}"
    if pd.api.types.is_numeric_dtype(series):
        return f"{series.min():g} a {series.max():g}"
    return "; ".join(sorted(series.dropna().astype(str).unique()))

data_dictionary = pd.DataFrame(dictionary_metadata)
data_dictionary["dtype_observado"] = data_dictionary["coluna"].map(
    lambda column: str(df[column].dtype)
)
data_dictionary["dominio_observado"] = data_dictionary["coluna"].map(
    lambda column: observed_domain(df[column])
)
data_dictionary["quantidade_ausentes"] = data_dictionary["coluna"].map(
    lambda column: int(df[column].isna().sum())
)

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)
dictionary_path = processed_dir / "dicionario_dados.csv"
data_dictionary.to_csv(dictionary_path, index=False, encoding="utf-8-sig")

print(f"Dicionário de dados salvo em: {dictionary_path.resolve()}")
display(data_dictionary)

## 2. Análise univariada

**Para que serve:** compreender cada variável isoladamente antes de procurar relações ou grupos. Histogramas revelam concentração, caudas e possíveis modos; boxplots resumem posição e extremos; barras mostram desbalanceamento categórico; a linha temporal revela mudanças de nível ao longo do ano.

As distribuições incluem o target apenas de forma descritiva. Os histogramas contínuos usam a regra de Freedman–Diaconis, cuja largura é

$$h = 2\,\operatorname{IQR}(x)\,n^{-1/3},$$

enquanto variáveis com significado discreto recebem intervalos semanticamente definidos. Os boxplots preservam a escala original e usam os limites $Q_1-1{,}5\,IQR$ e $Q_3+1{,}5\,IQR$ como referência visual para valores extremos.

In [ ]:
quantitative_columns = df.select_dtypes(include="number").columns.tolist()
categorical_columns = ["WeekStatus", "Day_of_week", TARGET_COLUMN]

print("Quantitativas:", quantitative_columns)
print("Qualitativas:", categorical_columns)
print("Temporal:", DATE_COLUMN)

In [ ]:
def histogram_bins(column: str, series: pd.Series):
    if column == "NSM":
        return np.arange(0, 86_400 + 3_600, 3_600)  # uma faixa por hora
    if column in {"Lagging_Current_Power_Factor", "Leading_Current_Power_Factor"}:
        return np.arange(0, 105, 5)  # intervalos de cinco pontos percentuais
    if column == "CO2(tCO2)":
        unique_values = np.sort(series.dropna().unique())
        step = np.diff(unique_values).min()
        return np.arange(unique_values.min() - step / 2, unique_values.max() + step, step)
    return np.histogram_bin_edges(series.dropna(), bins="fd")  # Freedman–Diaconis

fig, axes = plt.subplots(4, 2, figsize=(16, 16))
for ax, column in zip(axes.flat, quantitative_columns):
    show_kde = column not in {"CO2(tCO2)", "NSM"}
    sns.histplot(
        data=df,
        x=column,
        bins=histogram_bins(column, df[column]),
        kde=show_kde,
        ax=ax,
        color=PRIMARY_COLOR,
    )
    ax.set_title(f"Distribuição — {column}")
for ax in axes.flat[len(quantitative_columns):]:
    ax.set_visible(False)
fig.suptitle("Distribuições das variáveis quantitativas", fontsize=16, y=1.01)
fig.tight_layout()
save_figure(fig, "01_distribuicoes_quantitativas.png")
plt.show()

fig, axes = plt.subplots(4, 2, figsize=(16, 14))
for ax, column in zip(axes.flat, quantitative_columns):
    sns.boxplot(data=df, x=column, ax=ax, color=PRIMARY_COLOR)
    ax.set_title(f"Boxplot — {column}")
for ax in axes.flat[len(quantitative_columns):]:
    ax.set_visible(False)
fig.suptitle("Dispersão e valores extremos das variáveis quantitativas", fontsize=16, y=1.01)
fig.tight_layout()
save_figure(fig, "02_boxplots_quantitativos.png")
plt.show()

In [ ]:
daily_usage = df.set_index(DATE_COLUMN)["Usage_kWh"].resample("D").mean()

fig, ax = plt.subplots(figsize=(16, 5))
sns.lineplot(x=daily_usage.index, y=daily_usage.values, ax=ax, color=PRIMARY_COLOR)
ax.set(
    title="Consumo médio diário de energia",
    xlabel="Data",
    ylabel="Consumo médio (kWh)",
)
fig.tight_layout()
save_figure(fig, "03_consumo_medio_diario.png")
plt.show()

In [ ]:
category_orders = {
    "WeekStatus": ["Weekday", "Weekend"],
    "Day_of_week": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
    "Load_Type": ["Light_Load", "Medium_Load", "Maximum_Load"],
}

fig, axes = plt.subplots(1, 3, figsize=(21, 5))
for ax, column in zip(axes, categorical_columns):
    sns.countplot(data=df, x=column, order=category_orders[column], ax=ax, color=PRIMARY_COLOR)
    ax.set_title(f"Frequência — {column}")
    ax.set_xlabel(column)
    ax.set_ylabel("Quantidade de registros")
    ax.tick_params(axis="x", rotation=35)
    for container in ax.containers:
        ax.bar_label(container, fmt="%d")
fig.tight_layout()
save_figure(fig, "04_frequencias_categoricas.png")
plt.show()

## 3. Relações entre as features

**Para que serve:** identificar redundância, associação e comportamento conjunto entre features, ajudando a entender quais dimensões podem sustentar ou distorcer a segmentação.

Load_Type é removido desta etapa para que o target não determine a leitura exploratória das features. Pearson mede associação linear; hexbin e pairplot revelam forma e densidade; boxplots por calendário mostram mudanças condicionais; tabelas cruzadas examinam dependência entre categorias. Para duas variáveis $X$ e $Y$, a correlação de Pearson apresentada no mapa é

$$r_{XY}=\frac{\operatorname{cov}(X,Y)}{\sigma_X\sigma_Y}.$$

Valores próximos de $\pm1$ indicam associação linear forte; valores próximos de zero não descartam relações não lineares, motivo pelo qual também são usados pairplots, hexbin e perfis temporais.

In [ ]:
relationship_df = df.drop(columns=[TARGET_COLUMN])
relationship_numeric_columns = relationship_df.select_dtypes(include="number").columns
correlation_matrix = relationship_df[relationship_numeric_columns].corr(method="pearson")

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    ax=ax,
)
ax.set_title("Correlação entre variáveis quantitativas — target excluído")
fig.tight_layout()
save_figure(fig, "05_correlacao_variaveis_quantitativas.png")
plt.show()

In [ ]:
relationship_categorical_columns = ["WeekStatus", "Day_of_week"]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, column in zip(axes, relationship_categorical_columns):
    sns.boxplot(
        data=relationship_df,
        x=column,
        y="Usage_kWh",
        order=category_orders[column],
        ax=ax,
        color=PRIMARY_COLOR,
    )
    ax.set_title(f"Usage_kWh por {column}")
    ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
save_figure(fig, "06_consumo_por_variaveis_calendario.png")
plt.show()

weekday_status = pd.crosstab(
    relationship_df["Day_of_week"],
    relationship_df["WeekStatus"],
    normalize="index",
) * 100
weekday_status = weekday_status.reindex(category_orders["Day_of_week"])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(weekday_status, annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": "% por dia"}, ax=ax)
ax.set_title("Relação entre dia da semana e status da semana")
ax.set_xlabel("Status da semana")
ax.set_ylabel("Dia da semana")
fig.tight_layout()
save_figure(fig, "07_relacao_dia_status_semana.png")
plt.show()

In [ ]:
pairplot_sample = relationship_df[relationship_numeric_columns].sample(
    n=min(2_000, len(relationship_df)),
    random_state=42,
)
pair_grid = sns.pairplot(
    pairplot_sample,
    corner=True,
    diag_kind="hist",
    plot_kws={"s": 12, "alpha": 0.25, "color": PRIMARY_COLOR},
    diag_kws={"bins": "fd", "color": PRIMARY_COLOR},
)
pair_grid.figure.suptitle(
    "Relações pareadas entre variáveis quantitativas — amostra de 2.000 registros",
    y=1.01,
)
pair_grid.figure.tight_layout()
save_figure(pair_grid.figure, "08_relacoes_pareadas_quantitativas.png")
pair_grid.figure.suptitle(
    "Relações pareadas entre variáveis quantitativas — amostra de 2.000 registros",
    y=1.01,
)
pair_grid.figure.tight_layout()
save_figure(pair_grid.figure, "08_relacoes_pareadas_quantitativas.png")
plt.show()
save_figure(pair_grid.fig, "08_relacoes_pareadas_quantitativas.png")
plt.show()

relationship_pairs = [
    ("Usage_kWh", "CO2(tCO2)"),
    ("Usage_kWh", "Lagging_Current_Reactive.Power_kVarh"),
    ("Lagging_Current_Reactive.Power_kVarh", "Lagging_Current_Power_Factor"),
    ("Leading_Current_Reactive_Power_kVarh", "Leading_Current_Power_Factor"),
]
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, (x_column, y_column) in zip(axes.flat, relationship_pairs):
    density = ax.hexbin(
        relationship_df[x_column],
        relationship_df[y_column],
        gridsize=40,
        mincnt=1,
        cmap="viridis",
    )
    fig.colorbar(density, ax=ax, label="Quantidade de registros")
    ax.set_title(f"{y_column} × {x_column}")
    ax.set_xlabel(x_column)
    ax.set_ylabel(y_column)
fig.tight_layout()
save_figure(fig, "09_relacoes_bivariadas_hexbin.png")
plt.show()

In [ ]:
hourly_profile = (
    relationship_df.assign(hour=relationship_df["NSM"] // 3_600)
    .groupby(["hour", "WeekStatus"], as_index=False)["Usage_kWh"]
    .mean()
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.lineplot(
    data=hourly_profile,
    x="hour",
    y="Usage_kWh",
    hue="WeekStatus",
    marker="o",
    ax=ax,
)
ax.set(
    title="Perfil médio horário de consumo por status da semana",
    xlabel="Hora do dia",
    ylabel="Consumo médio (kWh)",
    xticks=range(24),
)
fig.tight_layout()
save_figure(fig, "10_perfil_horario_consumo.png")
plt.show()

### 3.1 Projeções multivariadas com PCA e t-SNE

**PCA — para que serve:** comprimir a matriz em eixos lineares ortogonais que preservam o máximo possível da variância global. Neste notebook, PCA permite visualizar estrutura ampla, quantificar variância explicada e projetar resultados dos modelos; ele não produz os três grupos.

**t-SNE — para que serve:** revelar vizinhanças locais que podem ficar ocultas em uma projeção linear. Neste notebook, ele funciona como verificação visual complementar em uma amostra reprodutível; seus eixos e distâncias globais não têm interpretação operacional e não são usados para definir K.

As duas projeções usam features numéricas padronizadas e features categóricas codificadas por one-hot. O target Load_Type e o identificador temporal não entram no ajuste.

No PCA, procura-se uma direção unitária $w$ que maximize a variância projetada:

$$w_1=\arg\max_{\lVert w\rVert=1}\operatorname{Var}(Xw).$$

O t-SNE preserva vizinhanças locais minimizando a divergência de Kullback–Leibler entre similaridades no espaço original e na projeção:

$$C=KL(P\Vert Q)=\sum_{i\neq j}p_{ij}\log\frac{p_{ij}}{q_{ij}}.$$

Como os eixos do t-SNE não têm interpretação física, o gráfico deve ser lido por proximidade e separação local, não por distância absoluta entre regiões afastadas.

In [ ]:
reduction_numeric_columns = quantitative_columns
reduction_categorical_columns = ["WeekStatus", "Day_of_week"]
reduction_features = reduction_numeric_columns + reduction_categorical_columns
assert TARGET_COLUMN not in reduction_features

reduction_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), reduction_numeric_columns),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), reduction_categorical_columns),
    ],
    verbose_feature_names_out=False,
)
transformed_features = reduction_preprocessor.fit_transform(df[reduction_features])

pca = PCA(n_components=2, random_state=42)
pca_coordinates = pca.fit_transform(transformed_features)
pca_df = pd.DataFrame(pca_coordinates, columns=["PCA_1", "PCA_2"])
pca_df["WeekStatus"] = df["WeekStatus"].to_numpy()

fig, ax = plt.subplots(figsize=(11, 7))
sns.scatterplot(
    data=pca_df,
    x="PCA_1",
    y="PCA_2",
    hue="WeekStatus",
    alpha=0.30,
    s=18,
    linewidth=0,
    ax=ax,
)
explained = pca.explained_variance_ratio_ * 100
ax.set(
    title=f"PCA — variância explicada acumulada: {explained.sum():.1f}%",
    xlabel=f"PCA 1 ({explained[0]:.1f}%)",
    ylabel=f"PCA 2 ({explained[1]:.1f}%)",
)
fig.tight_layout()
save_figure(fig, "11_projecao_pca.png")
plt.show()

In [ ]:
TSNE_SAMPLE_SIZE = min(3_000, len(df))
sample_positions = np.random.default_rng(42).choice(
    len(df),
    size=TSNE_SAMPLE_SIZE,
    replace=False,
)

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    max_iter=1_000,
    random_state=42,
)
tsne_coordinates = tsne.fit_transform(transformed_features[sample_positions])
tsne_df = pd.DataFrame(tsne_coordinates, columns=["tSNE_1", "tSNE_2"])
tsne_df["WeekStatus"] = df.iloc[sample_positions]["WeekStatus"].to_numpy()

fig, ax = plt.subplots(figsize=(11, 7))
sns.scatterplot(
    data=tsne_df,
    x="tSNE_1",
    y="tSNE_2",
    hue="WeekStatus",
    alpha=0.55,
    s=24,
    linewidth=0,
    ax=ax,
)
ax.set_title(f"t-SNE — amostra reprodutível de {TSNE_SAMPLE_SIZE:,} registros")
fig.tight_layout()
save_figure(fig, "12_projecao_tsne.png")
plt.show()

## 4. Diagnósticos adicionais

**Assimetria — para que serve:** medir direção e intensidade das caudas, indicando quando média e distância euclidiana podem ser influenciadas por concentração desigual.

**IQR — para que serve:** criar limites robustos para sinalização inicial de extremos, sem pressupor normalidade. Este bloco quantifica propriedades que os gráficos anteriores apenas sugerem e usa a faixa

$$[Q_1-1{,}5\,IQR,\;Q_3+1{,}5\,IQR].$$

Esses pontos não são removidos automaticamente: em consumo industrial, valores extremos podem representar operação legítima, mudança de regime ou anomalia.

In [ ]:
distribution_rows = []
for column in quantitative_columns:
    series = df[column].dropna()
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr
    outlier_count = int(((series < lower_limit) | (series > upper_limit)).sum())
    distribution_rows.append(
        {
            "variavel": column,
            "media": series.mean(),
            "mediana": series.median(),
            "desvio_padrao": series.std(),
            "assimetria": series.skew(),
            "curtose_excesso": series.kurt(),
            "iqr": iqr,
            "limite_inferior_iqr": lower_limit,
            "limite_superior_iqr": upper_limit,
            "quantidade_extremos_iqr": outlier_count,
            "percentual_extremos_iqr": 100 * outlier_count / len(series),
        }
    )

distribution_diagnostics = pd.DataFrame(distribution_rows).set_index("variavel")
distribution_diagnostics.to_csv(
    TABLES_DIR / "resumo_distribuicoes_quantitativas.csv",
    encoding="utf-8-sig",
)
display(distribution_diagnostics.round(3))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
skewness = distribution_diagnostics["assimetria"].sort_values()
skew_colors = [ANOMALY_COLOR if abs(value) > 1 else PRIMARY_COLOR for value in skewness]
axes[0].barh(skewness.index, skewness.values, color=skew_colors)
axes[0].axvline(0, color="black", linewidth=1)
axes[0].axvline(-1, color="gray", linestyle="--", linewidth=1)
axes[0].axvline(1, color="gray", linestyle="--", linewidth=1)
axes[0].set(title="Assimetria das distribuições", xlabel="Coeficiente de assimetria")
outlier_percentage = distribution_diagnostics["percentual_extremos_iqr"].sort_values()
axes[1].barh(outlier_percentage.index, outlier_percentage.values, color=ACCENT_COLOR)
axes[1].set(title="Valores extremos pela regra do IQR", xlabel="Percentual de registros (%)")
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.1f%%", padding=3)
sns.despine(fig=fig)
fig.tight_layout()
save_figure(fig, "13_diagnosticos_distribuicoes.png")
plt.show()

### 4.1 Associação monotônica e sazonalidade intradiária

**Spearman — para que serve:** detectar associação monotônica mesmo quando a relação não é linear ou quando a escala possui caudas e valores extremos. Ele é calculado sobre os postos das observações e complementa Pearson:

$$\rho_s=\operatorname{corr}(\operatorname{rank}(X),\operatorname{rank}(Y)).$$

**Mapa dia–hora — para que serve:** localizar janelas operacionais recorrentes e verificar se a separação em regimes pode estar associada ao calendário. Ele resume o consumo médio para cada combinação de dia da semana e hora.

In [ ]:
spearman_correlation = relationship_df[relationship_numeric_columns].corr(method="spearman")
weekday_hour_usage = (
    relationship_df.assign(hour=relationship_df["NSM"] // 3_600)
    .pivot_table(index="Day_of_week", columns="hour", values="Usage_kWh", aggfunc="mean")
    .reindex(category_orders["Day_of_week"])
)

fig, axes = plt.subplots(1, 2, figsize=(22, 8), gridspec_kw={"width_ratios": [1, 1.25]})
sns.heatmap(spearman_correlation, annot=True, fmt=".2f", cmap="vlag", center=0, square=True, ax=axes[0])
axes[0].set_title("Correlação de Spearman")
sns.heatmap(
    weekday_hour_usage,
    cmap="mako",
    cbar_kws={"label": "Consumo médio (kWh)"},
    ax=axes[1],
)
axes[1].set(title="Consumo médio por dia e hora", xlabel="Hora do dia", ylabel="Dia da semana")
fig.tight_layout()
save_figure(fig, "14_correlacao_spearman_e_mapa_dia_hora.png")
plt.show()

## 5. Segmentação exploratória em três regimes

Os algoritmos abaixo são instrumentos de EDA e recebem a mesma matriz transformada usada no PCA: variáveis numéricas padronizadas, calendário em one-hot e exclusão completa de Load_Type e date. O número de regimes é fixado em três pela cardinalidade conhecida do problema, sem usar a identidade das classes.

### Critérios de escolha e técnicas adiadas

**K-Means** é adotado como baseline porque o número de grupos é conhecido, o algoritmo escala bem para 35.040 registros e seus centróides permitem perfis operacionais simples. **GMM** é usado como contraponto probabilístico, pois admite sobreposição e dispersões distintas. **Isolation Forest** responde à parte de anomalias sem exigir rótulos. PCA oferece uma projeção linear interpretável, enquanto t-SNE é mantido apenas para vizinhanças locais e não define clusters.

O **cotovelo** e o **silhouette** não escolhem K nesta etapa: o cotovelo é visual e frequentemente subjetivo, enquanto maximizar silhouette pode produzir uma partição matematicamente compacta, mas incompatível com os três regimes estabelecidos pelo problema. Ambos permanecem como diagnósticos de consistência. AIC e BIC cumprem papel análogo no GMM e podem indicar subcomponentes estatísticos dentro dos três regimes.

DBSCAN e HDBSCAN não são usados porque deixam o número de grupos emergir da densidade e são sensíveis à escala, eps e min_samples; isso conflita com a hipótese explícita de três regimes. Agrupamento hierárquico e spectral clustering exigem matrizes de distância ou afinidade custosas para toda a base. K-Medoids seria mais robusto a extremos, mas tem custo maior e exigiria dependência adicional. SOM e autoencoders acrescentariam arquitetura e hiperparâmetros antes de consolidar o baseline. LOF e One-Class SVM ficam para comparação futura: o primeiro é fortemente local e o segundo escala pior neste volume.

Modelos supervisionados e métricas baseadas nas classes de Load_Type não são incluídos, pois antecipariam as etapas de modelagem e avaliação.

#### Métodos de densidade: DBSCAN e HDBSCAN

**DBSCAN — para que serve:** descobrir grupos de formato arbitrário e separar pontos em regiões pouco densas como ruído, sem informar previamente a quantidade de clusters. Ele define a vizinhança de uma observação por

$$N_{\varepsilon}(x_i)=\{x_j:d(x_i,x_j)\leq\varepsilon\},$$

e considera um ponto central quando $|N_{\varepsilon}(x_i)|\geq\text{min\_samples}$. Regiões conectadas por pontos centrais formam clusters; observações não alcançadas são tratadas como ruído. Essa propriedade seria útil se o objetivo fosse descobrir livremente regiões densas e separar ruído, sem conhecimento prévio da quantidade de regimes.

Ele não foi selecionado porque o número de grupos emerge dos parâmetros $\varepsilon$ e min_samples, enquanto o problema atual estabelece três regimes. Além disso, após padronização e one-hot, a matriz possui várias dimensões e distâncias tendem a se concentrar, dificultando a escolha de $\varepsilon$. Densidades diferentes entre períodos de baixa e alta carga também podem fazer um único valor de $\varepsilon$ unir grupos ou fragmentar regiões legítimas.

**HDBSCAN — para que serve:** encontrar grupos persistentes em diferentes níveis de densidade e atribuir probabilidades de pertencimento. Ele constrói uma hierarquia de densidades, reduzindo a dependência de um único $\varepsilon$, mas ainda determina livremente a quantidade de clusters, exige parâmetros como min_cluster_size e acrescenta uma dependência externa ao ambiente atual.

Esses métodos devem ser reconsiderados se a hipótese de três regimes for relaxada ou se o objetivo mudar para descoberta de micro-regimes e ruído. Nesse caso, a seleção deve incluir gráfico de k-distâncias, estabilidade em diferentes escalas e análise da proporção marcada como ruído.

#### Métodos hierárquicos, K-Medoids e agrupamento espectral

**Agrupamento hierárquico — para que serve:** revelar níveis de agrupamento e relações de inclusão entre macro e micro-regimes por meio de um dendrograma. A versão aglomerativa inicia com uma observação por grupo e realiza fusões sucessivas. No critério de Ward, a fusão escolhida minimiza o aumento da variância interna:

$$\Delta(A,B)=\frac{|A||B|}{|A|+|B|}\lVert\mu_A-\mu_B\rVert_2^2.$$

A técnica produz um dendrograma útil para investigar subgrupos dentro dos três regimes. Não foi aplicada à base completa porque requer armazenar ou manipular grande quantidade de distâncias, com custo de memória tipicamente quadrático em $n$. Para 35.040 registros, o dendrograma também perderia legibilidade. Uma aplicação futura pode usar amostragem estratificada no tempo ou os centróides obtidos pelo K-Means.

**K-Medoids — para que serve:** formar grupos representados por observações reais, reduzindo a influência de extremos e permitindo métricas de distância alternativas. Ele substitui centróides artificiais por medoides e minimiza

$$J=\sum_{i=1}^{n}\min_{m\in M}d(x_i,m).$$

Ele é mais robusto a valores extremos e aceita distâncias não euclidianas, mas é mais caro que K-Means e não está disponível diretamente no scikit-learn usado pelo projeto. Deve ser testado posteriormente se os extremos identificados pelo IQR alterarem excessivamente os centróides.

**Spectral clustering — para que serve:** separar grupos não convexos usando conectividade em um grafo, quando a distância direta a centróides não descreve bem a geometria. Ele constrói uma matriz de afinidade e utiliza autovetores do Laplaciano normalizado

$$L_{sym}=I-D^{-1/2}AD^{-1/2}.$$

Sua vantagem é separar estruturas não convexas. Entretanto, construir e decompor a matriz de afinidade para toda a base é custoso, e o resultado depende fortemente da definição de vizinhança ou do parâmetro do kernel. Ele só deve ser priorizado se as projeções e os diagnósticos mostrarem que os três regimes possuem geometrias claramente não convexas.

#### SOM, UMAP e autoencoders

**SOM — para que serve:** organizar dados multivariados em uma grade bidimensional que preserva proximidade topológica, facilitando a visualização de transições contínuas entre regimes. Self-Organizing Maps atualizam o neurônio vencedor e sua vizinhança:

$$w_j(t+1)=w_j(t)+\alpha(t)h_{cj}(t)[x(t)-w_j(t)].$$

SOM pode revelar continuidade entre regimes e produzir mapas fáceis de explorar, mas exige escolhas de tamanho da grade, função de vizinhança, taxa de aprendizagem e número de épocas. Esses graus de liberdade dificultariam atribuir diferenças ao dado ou à arquitetura antes de consolidar o baseline de três grupos.

**UMAP — para que serve:** criar uma projeção não linear que preserva vizinhanças locais e parte da estrutura global, geralmente com custo menor que t-SNE. Não foi incluído porque exigiria a dependência umap-learn e porque seus parâmetros n_neighbors e min_dist influenciam fortemente a aparência da projeção. Assim como t-SNE, uma projeção visual não deve ser tratada automaticamente como evidência de clusters.

**Autoencoders — para que servem:** aprender uma representação latente não linear e medir quão bem cada observação pode ser reconstruída, apoiando redução de dimensionalidade e anomalias. Eles minimizam erro de reconstrução, por exemplo

$$\mathcal{L}=\frac{1}{n}\sum_{i=1}^{n}\lVert x_i-g_\theta(f_\phi(x_i))\rVert_2^2.$$

Eles podem capturar relações não lineares e apoiar detecção de anomalias, mas introduzem arquitetura, regularização, inicialização, épocas e risco de aprender identidade ou reconstruir anomalias. Seu uso é mais apropriado depois que os baselines lineares e baseados em árvores estiverem estabelecidos.

#### Alternativas para detecção de anomalias

**LOF — para que serve:** identificar anomalias locais comparando a densidade de uma observação com a densidade de seus vizinhos, como um consumo incomum apenas dentro de determinado regime. Não foi selecionado nesta EDA porque depende de número de vizinhos e métrica, sofre com concentração de distâncias e seu escore é menos direto para aplicação futura em novos dados sem configurar o modo novelty.

**One-Class SVM — para que serve:** aprender uma fronteira de novidade ao redor da região considerada regular, inclusive com formas não lineares via kernel. Sua formulação depende de $\nu$, que controla a fração esperada de violações, e de parâmetros como $\gamma$. Pode modelar fronteiras complexas, mas tem maior custo de ajuste e sensibilidade a hiperparâmetros para 35.040 registros.

**Elliptic Envelope — para que serve:** delimitar uma região regular elipsoidal usando covariância robusta e distância de Mahalanobis. Como pressupõe distribuição aproximadamente gaussiana, as distribuições assimétricas e multimodais observadas tornam essa hipótese inadequada como primeira escolha. **Erro de reconstrução — para que serve:** sinalizar pontos que uma representação PCA ou autoencoder não consegue reproduzir bem; foi adiado porque exige definir um limiar e pode confundir um regime minoritário legítimo com anomalia.

Isolation Forest foi mantido como baseline por não depender de distância entre todos os pares, escalar bem e aceitar fronteiras não lineares. Em uma etapa posterior, LOF deve ser comparado dentro de cada um dos três regimes, evitando que diferenças legítimas entre regimes sejam interpretadas como anomalias.

### 5.1 K-Means

**Para que serve:** dividir as observações em três grupos compactos e produzir centróides que resumem perfis operacionais médios. Neste notebook, ele é o baseline principal por ser escalável, reproduzível e fácil de interpretar.

O K-Means procura centróides $\mu_k$ que minimizem a soma das distâncias quadráticas dentro dos grupos:

$$J=\sum_{k=1}^{K}\sum_{x_i\in C_k}\lVert x_i-\mu_k\rVert_2^2.$$

A partição principal usa $K=3$, obtido por len(df.Load_Type.unique()). **Cotovelo — para que serve:** observar quanto a inércia deixa de cair de forma relevante quando K aumenta, oferecendo uma referência visual de complexidade. **Silhouette — para que serve:** verificar simultaneamente se cada ponto está coeso com seu grupo e separado dos demais. Ambos são calculados entre $K=2$ e $K=8$ somente para conferir se a hipótese de três regimes apresenta estrutura geométrica razoável. Para cada observação, o silhouette compara a coesão $a(i)$ com a separação $b(i)$:

$$s(i)=\frac{b(i)-a(i)}{\max\{a(i),b(i)\}}.$$

Valores mais próximos de 1 indicam melhor separação. Mesmo que outro K maximize essa métrica, o notebook preserva os três regimes do problema e registra a divergência como evidência para investigação posterior.

In [ ]:
MODEL_SAMPLE_SIZE = min(10_000, len(df))
model_sample_positions = np.random.default_rng(42).choice(
    len(df), size=MODEL_SAMPLE_SIZE, replace=False
)
model_sample = transformed_features[model_sample_positions]

kmeans_rows = []
for n_clusters in range(2, 9):
    candidate = KMeans(n_clusters=n_clusters, n_init=20, random_state=42)
    candidate_labels = candidate.fit_predict(model_sample)
    kmeans_rows.append(
        {
            "n_clusters": n_clusters,
            "inercia": candidate.inertia_,
            "silhouette": silhouette_score(
                model_sample,
                candidate_labels,
                sample_size=min(3_000, MODEL_SAMPLE_SIZE),
                random_state=42,
            ),
        }
    )

kmeans_metrics = pd.DataFrame(kmeans_rows)
diagnostic_best_k = int(kmeans_metrics.loc[kmeans_metrics["silhouette"].idxmax(), "n_clusters"])
kmeans_metrics.to_csv(TABLES_DIR / "metricas_selecao_kmeans.csv", index=False, encoding="utf-8-sig")
display(kmeans_metrics.round(4))
print(f"K estrutural adotado: {EXPECTED_N_GROUPS}")
print(f"K com maior silhouette na amostra: {diagnostic_best_k} (apenas diagnóstico)")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.lineplot(data=kmeans_metrics, x="n_clusters", y="inercia", marker="o", ax=axes[0])
axes[0].axvline(EXPECTED_N_GROUPS, color=ANOMALY_COLOR, linestyle="--", label="Hipótese do problema: K=3")
axes[0].set(title="Cotovelo como diagnóstico", xlabel="Número de clusters (K)", ylabel="Inércia")
axes[0].legend()
sns.lineplot(data=kmeans_metrics, x="n_clusters", y="silhouette", marker="o", color=ACCENT_COLOR, ax=axes[1])
axes[1].axvline(EXPECTED_N_GROUPS, color=ANOMALY_COLOR, linestyle="--", label="Hipótese do problema: K=3")
axes[1].set(title="Silhouette como diagnóstico", xlabel="Número de clusters (K)", ylabel="Silhouette médio")
axes[1].legend()
sns.despine(fig=fig)
fig.tight_layout()
save_figure(fig, "15_selecao_kmeans.png")
plt.show()

In [ ]:
best_kmeans = KMeans(n_clusters=EXPECTED_N_GROUPS, n_init=30, random_state=42)
kmeans_labels = best_kmeans.fit_predict(transformed_features)
cluster_palette = sns.color_palette("colorblind", n_colors=EXPECTED_N_GROUPS)

kmeans_projection = pca_df.iloc[model_sample_positions].copy()
kmeans_projection["cluster"] = kmeans_labels[model_sample_positions].astype(str)
fig, ax = plt.subplots(figsize=(11, 7))
sns.scatterplot(
    data=kmeans_projection,
    x="PCA_1",
    y="PCA_2",
    hue="cluster",
    palette=cluster_palette,
    alpha=0.45,
    s=18,
    linewidth=0,
    ax=ax,
)
ax.set_title(f"K-Means projetado no PCA — K={EXPECTED_N_GROUPS} definido pelo problema")
fig.tight_layout()
save_figure(fig, "16_clusters_kmeans_pca.png")
plt.show()

cluster_profile_raw = (
    df.assign(cluster=kmeans_labels)
    .groupby("cluster")[quantitative_columns]
    .mean()
)
cluster_profile_z = (cluster_profile_raw - df[quantitative_columns].mean()) / df[quantitative_columns].std()
cluster_sizes = pd.Series(kmeans_labels).value_counts().sort_index().rename("quantidade")
cluster_profile_raw.join(cluster_sizes).to_csv(
    TABLES_DIR / "perfil_clusters_kmeans.csv",
    encoding="utf-8-sig",
)
profile_display_names = {
    "Usage_kWh": "Consumo (kWh)",
    "Lagging_Current_Reactive.Power_kVarh": "Reativa atrasada",
    "Leading_Current_Reactive_Power_kVarh": "Reativa adiantada",
    "CO2(tCO2)": "CO2",
    "Lagging_Current_Power_Factor": "FP atrasado",
    "Leading_Current_Power_Factor": "FP adiantado",
    "NSM": "Segundos do dia",
}
cluster_profile_display = cluster_profile_z.rename(columns=profile_display_names)
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(cluster_profile_display, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set(title="Perfil padronizado dos clusters K-Means", xlabel="Variável", ylabel="Cluster")
ax.tick_params(axis="x", rotation=25)
ax.tick_params(axis="y", rotation=0)
fig.tight_layout()
save_figure(fig, "17_perfil_clusters_kmeans.png")
plt.show()

### 5.2 Gaussian Mixture Model

**Para que serve:** representar os mesmos três regimes como distribuições probabilísticas sobrepostas, permitindo que fronteiras sejam graduais em vez de rígidas. Neste notebook, ele verifica se uma geometria elipsoidal e probabilística descreve a estrutura de forma diferente do K-Means.

O GMM representa a densidade como uma mistura probabilística de componentes gaussianos:

$$p(x)=\sum_{k=1}^{K}\pi_k\,\mathcal{N}(x\mid\mu_k,\Sigma_k),\qquad \sum_k\pi_k=1.$$

Diferentemente do K-Means, cada ponto possui probabilidades de pertencimento. O modelo principal também recebe três componentes, permitindo comparar duas geometrias sob a mesma hipótese de regimes. **AIC e BIC — para que servem:** comparar ajuste e complexidade, penalizando modelos com componentes adicionais; BIC aplica penalização mais forte conforme o tamanho da amostra. Aqui são diagnósticos: um mínimo em K maior pode representar subcomponentes de densidade, e não novos regimes operacionais. Para estabilidade computacional nesta EDA, é usada covariância diagonal.

In [ ]:
gmm_rows = []
for n_components in range(1, 9):
    candidate = GaussianMixture(
        n_components=n_components,
        covariance_type="diag",
        n_init=3,
        max_iter=300,
        random_state=42,
    )
    candidate.fit(model_sample)
    gmm_rows.append(
        {
            "n_componentes": n_components,
            "AIC": candidate.aic(model_sample),
            "BIC": candidate.bic(model_sample),
        }
    )

gmm_metrics = pd.DataFrame(gmm_rows)
diagnostic_bic_components = int(gmm_metrics.loc[gmm_metrics["BIC"].idxmin(), "n_componentes"])
gmm_metrics.to_csv(TABLES_DIR / "metricas_selecao_gmm.csv", index=False, encoding="utf-8-sig")
display(gmm_metrics.round(1))
print(f"Número estrutural de componentes adotado: {EXPECTED_N_GROUPS}")
print(f"Número com menor BIC: {diagnostic_bic_components} (apenas diagnóstico)")

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=gmm_metrics, x="n_componentes", y="AIC", marker="o", label="AIC", ax=ax)
sns.lineplot(data=gmm_metrics, x="n_componentes", y="BIC", marker="o", label="BIC", ax=ax)
ax.axvline(EXPECTED_N_GROUPS, color=ANOMALY_COLOR, linestyle="--", label="Hipótese do problema: K=3")
ax.axvline(diagnostic_bic_components, color="gray", linestyle=":", label=f"BIC mínimo = {diagnostic_bic_components}")
ax.set(title="AIC e BIC como diagnósticos do GMM", xlabel="Número de componentes", ylabel="Critério de informação")
ax.legend()
sns.despine(fig=fig)
fig.tight_layout()
save_figure(fig, "18_selecao_gmm.png")
plt.show()

best_gmm = GaussianMixture(
    n_components=EXPECTED_N_GROUPS,
    covariance_type="diag",
    n_init=5,
    max_iter=300,
    random_state=42,
)
gmm_labels = best_gmm.fit_predict(transformed_features)
gmm_projection = pca_df.iloc[model_sample_positions].copy()
gmm_projection["componente"] = gmm_labels[model_sample_positions].astype(str)
fig, ax = plt.subplots(figsize=(11, 7))
sns.scatterplot(
    data=gmm_projection,
    x="PCA_1",
    y="PCA_2",
    hue="componente",
    palette=sns.color_palette("husl", n_colors=EXPECTED_N_GROUPS),
    alpha=0.45,
    s=18,
    linewidth=0,
    ax=ax,
)
ax.set_title(f"Componentes do GMM no PCA — K={EXPECTED_N_GROUPS} definido pelo problema")
fig.tight_layout()
save_figure(fig, "19_componentes_gmm_pca.png")
plt.show()

### 5.3 Isolation Forest

**Para que serve:** encontrar medições globalmente incomuns sem precisar de exemplos rotulados de anomalias. Neste notebook, ele complementa a segmentação: primeiro descrevem-se três regimes; depois sinalizam-se pontos raros que podem ocorrer dentro ou fora deles.

Isolation Forest procura observações que exigem poucos cortes aleatórios para serem isoladas. Seu escore clássico depende do comprimento médio de caminho $E[h(x)]$:

$$s(x,n)=2^{-E[h(x)]/c(n)}.$$

Escores altos sugerem maior facilidade de isolamento. O algoritmo é complementar aos três grupos: ele sinaliza observações incomuns sem criar uma quarta classe ou usar Load_Type. Foi preferido nesta EDA por escalar bem, aceitar relações não lineares e produzir um escore global. A taxa de contaminação de 2% usada abaixo é uma hipótese exploratória, não uma estimativa comprovada da incidência real de anomalias.

In [ ]:
CONTAMINATION_RATE = 0.02
isolation_forest = IsolationForest(
    n_estimators=300,
    contamination=CONTAMINATION_RATE,
    max_samples="auto",
    n_jobs=-1,
    random_state=42,
)
isolation_labels = isolation_forest.fit_predict(transformed_features)
anomaly_score = -isolation_forest.decision_function(transformed_features)
is_anomaly = isolation_labels == -1

anomaly_records = df.drop(columns=[TARGET_COLUMN]).copy()
anomaly_records["anomaly_score"] = anomaly_score
anomaly_records["is_anomaly"] = is_anomaly
anomaly_records.loc[is_anomaly].sort_values("anomaly_score", ascending=False).to_csv(
    TABLES_DIR / "anomalias_isolation_forest.csv",
    index=False,
    encoding="utf-8-sig",
)
print(f"Registros sinalizados: {is_anomaly.sum():,} ({is_anomaly.mean():.2%})")
display(anomaly_records.loc[is_anomaly].nlargest(10, "anomaly_score"))

normal_positions = np.flatnonzero(~is_anomaly)
normal_plot_positions = np.random.default_rng(42).choice(
    normal_positions,
    size=min(8_000, len(normal_positions)),
    replace=False,
)
daily_anomaly_count = (
    pd.DataFrame({"date": df[DATE_COLUMN], "is_anomaly": is_anomaly})
    .set_index("date")["is_anomaly"]
    .resample("D")
    .sum()
)

fig, axes = plt.subplots(1, 2, figsize=(19, 7))
axes[0].scatter(
    pca_coordinates[normal_plot_positions, 0],
    pca_coordinates[normal_plot_positions, 1],
    s=10,
    alpha=0.18,
    color="gray",
    label="Regular (amostra)",
)
axes[0].scatter(
    pca_coordinates[is_anomaly, 0],
    pca_coordinates[is_anomaly, 1],
    s=24,
    alpha=0.75,
    color=ANOMALY_COLOR,
    label="Sinalizado",
)
axes[0].set(title="Isolation Forest projetado no PCA", xlabel="PCA 1", ylabel="PCA 2")
axes[0].legend()
sns.lineplot(x=daily_anomaly_count.index, y=daily_anomaly_count.values, color=ANOMALY_COLOR, ax=axes[1])
axes[1].set(title="Quantidade diária de sinais de anomalia", xlabel="Data", ylabel="Registros sinalizados")
sns.despine(fig=fig)
fig.tight_layout()
save_figure(fig, "20_anomalias_isolation_forest.png")
plt.show()